# 12. Lecke - Csevegési Előzmények Csökkentése Agent Scratchpaddel

Ez a jegyzetfüzet azt mutatja be, hogyan lehet kezelni a kontextust hosszú beszélgetések során a Microsoft Agent Framework segítségével. Ahogy a beszélgetések nőnek, a tokenek száma is növekszik — végül meghaladva a modell kontextusablakát. Ezt egy **kontextus összefoglalási mintával** és egy **agent scratchpaddel** oldjuk meg a tartós memória érdekében.

## Amit megtanulsz:
1. **Miért fontos a kontextus kezelése**: A tokenkorlátok és a kontextusablakok megértése
2. **Kontextusérzékeny ügynökök**: Olyan ügynökök építése, amelyek kezelik saját beszélgetési kontextusukat
3. **Kontextus összefoglalási minta**: Eszközök használata a beszélgetési előzmények tömörítésére
4. **Agent Scratchpad**: Tartós memória, amely túléli a kontextus csökkentését

## Előfeltételek:
- Azure OpenAI beállítás környezeti változókkal konfigurálva
- Az előző leckékből származó alapvető ügynökös fogalmak megértése


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Miért fontos a kontextuskezelés

Minden LLM-nek véges méretű **kontekstusablaka** van — az a maximális token-szám, amelyet egyetlen kérésben feldolgozni tud. Ahogy egy többszörös fordulós beszélgetés halad előre:

- A **tokenek száma lineárisan növekszik** minden felhasználói üzenettel és segítő válasszal.
- A **prompt tokenek dominálják a költséget**, mert a teljes előzményt minden fordulóban újraküldik.
- Végül a beszélgetés **túllépi a kontextusablakot**, és a modell vagy levágja az adatot, vagy hibát jelez.

### Stratégiák a kontextus kezelésére

| Stratégia | Működése | Kereszthaszon |
|---|---|---|
| **Levágás** | Legöregebb üzenetek eldobása | Korai kontextus elveszik |
| **Összefoglalás** | Régebbi üzenetek összegzése | Néhány részlet elveszik, de a fő pontok megmaradnak |
| **Jegyzetfüzet / Külső memória** | Fontos tények tárolása a beszélgetésen kívül | Eszközhívás szükséges, de túléli bármilyen csökkentést |

Ebben a jegyzetfüzetben kombináljuk az **összefoglalást** egy **jegyzetfüzet eszközzel**, hogy az ügynök folytonosságot tudjon fenntartani még akkor is, ha a beszélgetési előzmény tömörítésre kerül.


## Kontextusérzékeny ügynök létrehozása


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Hosszú beszélgetés szimulálása

Nézzünk meg egy többfordulós beszélgetést, hogy lássuk, hogyan halmozódik fel a kontextus. Az ügynöknek meg kell őriznie a kulcsfontosságú részleteket (preferenciák, költségvetés, utazási dátumok) a fordulók során, és folyamatosnak kell mutatkoznia.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Figyeld meg, hogyan őrzi meg az ügynök a korábbi fordulók kontextusát — tud Japánról, sushiról, templomokról, fényképezésről, a 3000 dolláros költségvetésről, egyéni utazásról és az áprilisi időszakról. Egy rövid beszélgetés során ez jól működik, de ahogy a beszélgetés hosszabb lesz, az egész előzmény újbóli elküldése erőforrás-igényessé válik.

Folytassuk a beszélgetést további fordulókkal, hogy lássuk a kontextus felhalmozódását:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Kontextus Összefoglaló Minta

Ahogy a beszélgetés nő, használhatunk egy **összefoglaló eszközt**, hogy a felhalmozott kontextust tömör formátumba sűrítsük. Az ügynök ezt az eszközt hívja meg, hogy rögzítse a fontosabb preferenciákat, így még akkor is, ha a régebbi üzenetek törlődnek, a lényeges információk megmaradnak.

Ez a minta az összetettebb előzménytömörítés alapköve:
1. Az ügynök azonosítja a beszélgetés kulcsfontosságú tényeit
2. Meghívja az összefoglaló eszközt, hogy ezeket megőrizze
3. A régebbi üzenetek biztonságosan eltávolíthatók, mert az összefoglaló rögzíti a lényegeseket

Lent definiálunk egy `summarize_preferences` eszközt, amelyet az ügynök hívhat meg, hogy tömör összefoglalót rögzítsen az általa tanultakról.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Összefoglaló

Ebben a leckében megtanultad, hogyan kezelheted a kontextust hosszú üzemidejű ügynöki beszélgetések során a Microsoft Agent Framework segítségével:

### Kulcsfogalmak
- **A kontextusablakok végesek** — a beszélgetés előzményeiben szereplő minden token pénzbe kerül és beleszámít a korlátba.
- **Összefoglaló eszközök** lehetővé teszik az ügynök számára, hogy a felhalmozott kontextust tömör összefoglalóvá sűrítse, csökkentve a tokenhasználatot miközben megőrzi a lényeges információkat.
- **Ügynöki jegyzetek (scratchpadek)** tartós külső memóriát biztosítanak, amelyek túlélnek bármilyen beszélgetés csökkentést.

### Amit elkészítettél
- Egy **kontextusérzékeny ügynököt**, amely fenntartja a folytonosságot többfordulós beszélgetések során
- Egy **összefoglaló eszközt** (`summarize_preferences`), amely tömör formában rögzíti a kulcsfontosságú felhasználói adatokat
- Egy **többfordulós beszélgetést**, amely bemutatja a kontextus megőrzését és a változások kezelését

### Valós alkalmazások
- **Ügyfélszolgálati robotok**: megjegyzik a preferenciákat hosszú támogatási ülések során
- **Személyi asszisztensek**: követik a folyamatban lévő projekteket kontextus újramesélés nélkül
- **Oktatási tutorok**: fenntartják a diákok előrehaladását sok interakció során

### Következő lépések
- Teljes jegyzeteszköz megvalósítása fájl-alapú perzisztenciával
- Automatikus előzmény-csonkítás hozzáadása összefoglalás után
- Vektoradatbázisokkal való kombinálás szemantikus memória kereséshez
- Olyan ügynökök készítése, amelyek napokkal később is képesek folytatni a beszélgetést teljes kontextussal


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ezt a dokumentumot az AI fordító szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével fordítottuk le. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum a saját nyelvén tekintendő hiteles forrásnak. Kritikus információk esetén professzionális emberi fordítást javaslunk. Nem vállalunk felelősséget az ebből a fordításból eredő félreértésekért vagy hibás értelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
